In [ ]:
# Pull gentrification data as timeseries from Census

import pandas as pd
from census import Census

API_KEY = "6c68cfec142f0cf6749e8e0775e83b97dd60491c"
c = Census(API_KEY)

STATE_FIPS = "08"   # Colorado
COUNTY_FIPS = "031"  # Denver County

variables = {
    "B19013_001E": "median_household_income",
    "B25064_001E": "median_gross_rent",
    "B25077_001E": "median_home_value",
    "B03002_001E": "total_pop",
    "B03002_003E": "white_nonhispanic",
    "B03002_004E": "black_nonhispanic",
    "B03002_012E": "hispanic_latino",
}
fields = ("NAME",) + tuple(variables.keys())

years = range(2011, 2023)  # bump upper bound as newer ACS5 vintages release

all_years = []
for year in years:
    try:
        result = c.acs5.state_county_tract(
            fields, STATE_FIPS, COUNTY_FIPS, Census.ALL, year=year
        )
    except Exception as e:
        print(f"{year}: failed -- {e}")
        continue
    df = pd.DataFrame(result).rename(columns=variables)
    df["year"] = year
    all_years.append(df)

acs_tracts = pd.concat(all_years, ignore_index=True)

# Census encodes suppressed/not-applicable values as large negative sentinels
numeric_cols = list(variables.values())
for col in numeric_cols:
    acs_tracts[col] = pd.to_numeric(acs_tracts[col], errors="coerce")
    acs_tracts.loc[acs_tracts[col] < 0, col] = pd.NA

acs_tracts["GEOID"] = acs_tracts["state"] + acs_tracts["county"] + acs_tracts["tract"]

# --------------------------------------------------------------------------
# Tract boundary vintage: rather than crosswalking 2010-vintage tracts to
# 2020-vintage tracts, just flag which boundary set each year's GEOIDs
# belong to.
# --------------------------------------------------------------------------
acs_tracts["boundary_vintage"] = acs_tracts["year"].apply(
    lambda y: "2020_tracts" if y >= 2020 else "2010_tracts"
)

In [4]:
# --------------------------------------------------------------------------
# Inflation adjustment: pull CPI-U (U.S. city average, all items) directly
# from FRED so this doesn't go stale as more years are added later.
# --------------------------------------------------------------------------
cpi = pd.read_csv("/Users/evanfarrenkopf/Desktop/Projects/Airbnb/data/CPIAUCNS.csv")
cpi["observation_date"] = pd.to_datetime(cpi["observation_date"])
cpi["year"] = cpi["observation_date"].dt.year
annual_cpi = cpi.groupby("year")["CPIAUCNS"].mean().rename("cpi")

BASE_YEAR = max(years)  # express everything in the most recent year's dollars
base_cpi = annual_cpi.loc[BASE_YEAR]

acs_tracts = acs_tracts.merge(annual_cpi, on="year", how="left")
acs_tracts["inflation_factor"] = base_cpi / acs_tracts["cpi"]

for col in ["median_household_income", "median_gross_rent", "median_home_value"]:
    acs_tracts[f"real_{col}"] = acs_tracts[col] * acs_tracts["inflation_factor"]

print(acs_tracts.shape)
print(acs_tracts["boundary_vintage"].value_counts())

(1830, 19)
boundary_vintage
2010_tracts    1296
2020_tracts     534
Name: count, dtype: int64


In [5]:
acs_tracts.to_csv("data/denver_tract_acs_2011_2022.csv", index=False)

In [ ]:
import geopandas as gpd
import pandas as pd
import pygris

neighborhoods = gpd.read_file("/Users/evanfarrenkopf/Desktop/Projects/Airbnb/data/denver_neighborhoods.geojson") 

acs_tracts = pd.read_csv("data/denver_tract_acs_2011_2022.csv", dtype={"GEOID": str})

# Boundaries only change once per vintage, so one geometry pull covers every
# year within that vintage
tracts_2010v = pygris.tracts(state="CO", county="031", year=2015, cb=True)[["GEOID", "geometry"]]
tracts_2020v = pygris.tracts(state="CO", county="031", year=2021, cb=True)[["GEOID", "geometry"]]

def attach_geometry(acs_df, tract_geo, vintage_label):
    subset = acs_df[acs_df["boundary_vintage"] == vintage_label]
    merged = subset.merge(tract_geo, on="GEOID", how="left")
    return gpd.GeoDataFrame(merged, geometry="geometry", crs=tract_geo.crs)

gdf_2010v = attach_geometry(acs_tracts, tracts_2010v, "2010_tracts")
gdf_2020v = attach_geometry(acs_tracts, tracts_2020v, "2020_tracts")

Using FIPS code '08' for input 'CO'
Using FIPS code '08' for input 'CO'


In [ ]:
def interpolate_to_neighborhoods(tract_gdf, neighborhoods, sum_cols, avg_cols,
                                   weight_col="total_pop", id_col="NBHD_ID"):
    # Reproject to a projected CRS before computing area -- lat/long degrees
    # produce meaningless area values. Colorado State Plane or a CONUS
    # equal-area projection (EPSG:5070) both work.
    tract_gdf = tract_gdf.to_crs(5070)
    neighborhoods = neighborhoods.to_crs(5070)
    tract_gdf["tract_area"] = tract_gdf.geometry.area

    overlay = gpd.overlay(tract_gdf, neighborhoods, how="intersection")
    overlay["area_frac"] = overlay.geometry.area / overlay["tract_area"]

    # Counts: allocate proportional to the share of the tract's area this
    # piece covers, then sum the pieces feeding into each neighborhood
    for col in sum_cols:
        overlay[f"{col}_alloc"] = overlay[col] * overlay["area_frac"]

    # Medians: population-weight each tract piece, then take a weighted
    # average across all pieces feeding into a neighborhood
    overlay["piece_weight"] = overlay["area_frac"] * overlay[weight_col]
    for col in avg_cols:
        overlay[f"{col}_x_weight"] = overlay[col] * overlay["piece_weight"]

    grouped = overlay.groupby(["year", id_col])
    out = grouped[[f"{c}_alloc" for c in sum_cols]].sum()
    out.columns = sum_cols

    weight_sum = grouped["piece_weight"].sum()
    for col in avg_cols:
        out[col] = grouped[f"{col}_x_weight"].sum() / weight_sum

    return out.reset_index()

sum_cols = ["total_pop", "white_nonhispanic", "black_nonhispanic", "hispanic_latino"]
avg_cols = ["real_median_household_income", "real_median_gross_rent", "real_median_home_value"]

neigh_2010v = interpolate_to_neighborhoods(gdf_2010v, neighborhoods, sum_cols, avg_cols)
neigh_2010v["boundary_vintage"] = "2010_tracts"

neigh_2020v = interpolate_to_neighborhoods(gdf_2020v, neighborhoods, sum_cols, avg_cols)
neigh_2020v["boundary_vintage"] = "2020_tracts"

neighborhood_timeseries = pd.concat([neigh_2010v, neigh_2020v], ignore_index=True)

name_lookup = (
    neighborhoods[["NBHD_ID", "NBHD_NAME"]]
    .drop_duplicates()
    .rename(columns={"NBHD_NAME": "neighborhood"})
)

neighborhood_timeseries = neighborhood_timeseries.merge(name_lookup, on="NBHD_ID", how="left")

cols = ["neighborhood"] + [c for c in neighborhood_timeseries.columns if c != "neighborhood"]
neighborhood_timeseries = neighborhood_timeseries[cols]

neighborhood_timeseries.to_csv("data/denver_neighborhood_gentrification_timeseries.csv", index=False)

In [13]:
print(neighborhood_timeseries["neighborhood"].isna().sum())

0
